In [1]:
# Load Data 
df = spark.sql("SELECT * FROM lh_SalesAnalytics.bronze.CustomerChurn")
display(df)

StatementMeta(, 50802368-edea-4275-9d5e-f7d7d905fb95, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 996f75c3-022f-401a-b7f3-a6e7db73c739)

In [12]:
dict(df.dtypes)['cr8b0_monthlycharges']

StatementMeta(, 50802368-edea-4275-9d5e-f7d7d905fb95, 16, Finished, Available, Finished, False)

'decimal(38,18)'

In [15]:
from pyspark.sql import functions as F

# Data Cleaning Functions
def clean_data(df):

        
    # 1. Delete Duplicates
    df = df.drop_duplicates()

    # 2. Define Columns where nan values has to be deleted (under 5%)
    drop_na_cols = [
        'cr8b0_recordid', 'cr8b0_paymentmethod', 'cr8b0_contracttype',
        'cr8b0_customerid', 'cr8b0_internetservicetype', 
        'cr8b0_monthlycharges', 'cr8b0_totalcharges_base', 'cr8b0_gender'
    ]
    df = df.dropna(subset=drop_na_cols)
    
    # 3. fill missing values with 0
    bool_na_cols = [c for c, t in df.dtypes if t in ('boolean', 'int')]
    df = df.fillna(value=0, subset=bool_na_cols)


    # 4. change decimal format
    decimal_cols = [c for c, t in df.dtypes if "decimal" in t]

    # oop for changing datatype
    for col_name in decimal_cols:
        df = df.withColumn(col_name, F.col(col_name).cast("decimal(12,2)"))

    # 4. Delete unused Columns
    df = df.drop('cr8b0_monthlycharges_base','cr8b0_totalcharges_base', 'createdby', 'createdbyname', 'createdbyyominame', 'createdon', 
                 'createdonbehalfby', 'createdonbehalfbyname', 'createdonbehalfbyyominame', 
                 'exchangerate', 'importsequencenumber', 'modifiedby', 'modifiedbyname', 'modifiedbyyominame',
                 'modifiedon', 'modifiedonbehalfby', 'modifiedonbehalfbyname', 'modifiedonbehalfbyyominame', 
                 'overriddencreatedon', 'ownerid', 'owneridname', 'owneridtype', 'owneridyominame', 'owningbusinessunit',
                 'owningbusinessunitname', 'owningteam', 'owninguser', 'statecode', 'statuscode', 'timezoneruleversionnumber', 
                 'transactioncurrencyid', 'transactioncurrencyidname', 'utcconversiontimezonecode', 'versionnumber'
                )
    
    return df

StatementMeta(, 50802368-edea-4275-9d5e-f7d7d905fb95, 19, Finished, Available, Finished, False)

In [17]:
# Write cleaned data into a temporary file to be validated by Great Expectations after
df_clean = clean_data(df)
df_clean.write.format("delta").mode("overwrite").save("Files/bronze/cleaned_data")
display(df_clean)

StatementMeta(, 50802368-edea-4275-9d5e-f7d7d905fb95, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0bb0495-45c3-4ed9-a56f-3329d65538b5)